<a href="https://colab.research.google.com/github/ViKing-Coder-jpg/CryptoMoon/blob/main/BackEnd/CryptoMoon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CryptoMoon -A Bitcoin Predictor


# Data Extraction and Library Importing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

In [ ]:
import kagglehub
path = kagglehub.dataset_download("aiwithcagri/bitcoin-12-years-price-january-2026")
data= pd.read_csv(path+'/bitcoin (1).csv')
data.shape

Using Colab cache for faster access to the 'bitcoin-12-years-price-january-2026' dataset.


(4134, 6)

#Data Cleaning and Preprocessing

In [ ]:
print('Empty Cells:',data.isna().sum().sum())
data['Date']=pd.to_datetime(data['Date'])
data.sort_values('Date',inplace=True)
data['Close'].median()


Empty Cells: 0


10795.1455078125

#Feature Engineering and train-test splitting

In [ ]:


featData=pd.DataFrame()
featData['Close']=data['Close']
featData['Target']=data['Close'].shift(-1)
featData['SMA_20']=data['Close'].rolling(window=20).mean()
featData['SMA_50']=data['Close'].rolling(window=50).mean()
featData['EMA_12'] = data['Close'].ewm(span=12, adjust=False).mean()
featData['EMA_26'] = data['Close'].ewm(span=26, adjust=False).mean()
featData['MACD'] = featData['EMA_12'] - featData['EMA_26']

delta = data['Close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
featData['RSI'] = 100 - (100 / (1 + rs))

featData.dropna(inplace=True)



In [ ]:
y=featData['Target']
X=featData.drop('Target',axis=1)


tscv = TimeSeriesSplit(n_splits=5)

# Model Training and Testing
  Using XGBoost as Base ML Model



In [ ]:

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    model=XGBRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print('Mean Squared Error:',mean_squared_error(y_test,y_pred))
    print('Root Mean Squared Error:',root_mean_squared_error(y_test,y_pred))
    print('R2 Score:',r2_score(y_test,y_pred))
    print()


Mean Squared Error: 37501294.46874136
Root Mean Squared Error: 6123.830048975997
R2 Score: -1.039220537798029

Mean Squared Error: 348093.9103455039
Root Mean Squared Error: 589.9948392532801
R2 Score: 0.9342355753888905

Mean Squared Error: 742406991.1419058
Root Mean Squared Error: 27247.146477051607
R2 Score: -1.5727898751809484

Mean Squared Error: 7432404.384857206
Root Mean Squared Error: 2726.2436400397537
R2 Score: 0.9036207492543469

Mean Squared Error: 1220801771.6962945
Root Mean Squared Error: 34939.97383651417
R2 Score: -2.1014473056468517



Mean Squared Error: 1220801771.6962945
Root Mean Squared Error: 34939.97383651417
R2 Score: -2.1014473056468517
